# Giải thích `Core/S2.hpp` của LIMOncello

Notebook này giải thích file `S2.hpp` dùng để xử lý **gravity trên manifold \(S^2\)** trong LIMOncello.

Mục tiêu của file này là nối hai thế giới lại với nhau:

- **state lưu gravity như một vector 3D** trong `State.hpp`
- nhưng **error-state của gravity chỉ có 2 bậc tự do** vì đầu mút của gravity bị buộc nằm trên mặt cầu \(S^2(r)\)

Vì vậy, `S2.hpp` chính là lớp "đổi tọa độ" giữa:

- **biểu diễn ambient 3D**: một vector \(x \in \mathbb{R}^3\) với chuẩn cố định
- **biểu diễn tangent 2D**: một increment \(u \in \mathbb{R}^2\) trên mặt phẳng tiếp tuyến tại \(x\)

Notebook này bám theo:

1. **Appendix A của paper LIMOncello**, đặc biệt Eq. (21)–(24), là nơi paper định nghĩa \(S^2\), \(x \oplus \tau\), \(y \ominus x\) và các Jacobian gần kỳ dị.
2. **`State.hpp`**, là nơi các hàm `S2::boxplus`, `S2::oplus`, `S2::ominus` thật sự được dùng trong `predict()` và `update()`.

Kết luận ngắn gọn cần nhớ trước:

> `S2.hpp` không phải "một bộ lọc khác". Nó chỉ là **bộ công cụ hình học** để gravity được cập nhật **đúng trên mặt cầu** thay vì cộng trừ như vector Euclid thường.

## 1. Bức tranh lớn trong LIMOncello

Paper LIMOncello định nghĩa state trên manifold

\[
\mathcal M = SGal(3)\times SE(3)\times \mathbb R^6 \times S^2.
\]

Trong đó:

- \(SGal(3)\): pose + velocity + time của IMU
- \(SE(3)\): extrinsics LiDAR–IMU
- \(\mathbb R^6\): gyro bias và accel bias
- \(S^2\): gravity có **hướng thay đổi được** nhưng **độ lớn giữ cố định**

Trong `State.hpp`, gravity lại được lưu bằng `manif::R3`, tức là **vector 3D thường**. Nhưng covariance không dùng đủ 3 chiều cho gravity, mà dùng **2 chiều tangent**:

```cpp
static constexpr int DoF = BundleT::DoF;   // full state
static constexpr int DoFS2 = DoF - 1;      // gravity is treated as S2
...
P.diagonal().segment(22, 2).setConstant(cfg.ikfom.covariance.initial_cov.gravity);
```

Đây là lý do `S2.hpp` tồn tại: file này cung cấp các phép toán để chuyển qua lại giữa:

- vector gravity 3D đang lưu trong state
- increment 2D của Kalman filter

## 2. Ký hiệu sẽ dùng trong notebook

Ta dùng đúng tinh thần của code:

- \(x, y \in \mathbb R^3\): hai vector có chuẩn cố định, nằm trên mặt cầu \(S^2(r)\)
- \(u, \tau \in \mathbb R^2\): increment trong mặt phẳng tiếp tuyến
- \(R(\phi) = \mathrm{Exp}_{SO(3)}(\phi)\): ma trận quay từ vector trục-góc \(\phi\in\mathbb R^3\)
- \(B(x)\in\mathbb R^{3\times 2}\): cơ sở của mặt phẳng tiếp tuyến tại \(x\)

Điểm rất quan trọng:

- **paper** Appendix A dùng ký hiệu \(\oplus,\ominus\) cho các phép toán trên \(S^2\)
- **file này** có thêm một helper tên `boxplus(x, y3)` nhận increment 3D; helper này **không phải** phép \(x\oplus \tau\) 2D trong Appendix A

Nói ngắn gọn:

- `boxplus(x, y3)` = quay vector 3D `x` bởi increment quay 3D `y3`
- `oplus(x, u2)` = cập nhật 1 điểm trên \(S^2\) bằng increment 2D trên tangent plane
- `ominus(y, x)` = lấy sai khác 2D giữa hai điểm trên \(S^2\)

## 3. Đoạn code lõi của `S2.hpp`

Dưới đây là phần cốt lõi của file, rút gọn cho dễ đọc:

```cpp
inline Matrix3d R(const Vector3d& x) {
  return manif::SO3d::Tangent(x).exp().rotation();
}

Vector3d boxplus(const Vector3d& x, const Vector3d& y,
                 Opt<Matrix3d> J_x = std::nullopt,
                 Opt<Matrix3d> J_y = std::nullopt) {
  Matrix3d Ry = R(y);
  Vector3d out = Ry * x;

  if (J_x) *J_x = Ry;
  if (J_y) *J_y = -Ry * manif::skew(x) * manif::SO3d::Tangent(y).rjac();
  return out;
}
```

```cpp
inline Matrix3x2d B(const Vector3d& x) {
  Vector3d   e_i  = Vector3d::UnitZ() * -1.;
  Matrix3x2d E_jk = Matrix3d::Identity().leftCols<2>() * -1.;
  ...
  out = R(axis*theta) * E_jk;
  return out / x.norm();
}
```

```cpp
inline Vector3d oplus(const Vector3d& x, const Vector2d& u,
                      Opt<Matrix3d> J_x = std::nullopt,
                      Opt<Matrix3x2d> J_u = std::nullopt) {
  Matrix3d RBu = R(B(x)*u);
  Vector3d out = tinyu ? x : (RBu*x).eval();

  if (J_x) { assert(tinyu); *J_x = Matrix3d::Identity(); }
  if (J_u) {
    if (tinyu)
      *J_u = -manif::skew(x)*B(x);
    else
      *J_u = -RBu * manif::skew(x) * manif::SO3d::Tangent(B(x)*u).rjac() * B(x);
  }
  return out; 
}
```

```cpp
inline Vector2d ominus(const Vector3d& y, const Vector3d& x,
                       Opt<Matrix2x3d> J_y = std::nullopt) {
  Vector3d cross      = x.cross(y);
  double   cross_norm = cross.norm();
  double   dot        = x.dot(y);

  if (cross_norm >= 1e-12) {
    Vector3d axis  = cross / cross_norm;
    double   theta = std::atan2(cross_norm, dot);
    out = B(x).transpose() * (axis*theta);
    ...
  } else {
    out = dot >= 0. ? Vector2d::Zero() : (M_PI*Vector2d::UnitX()).eval();
    ...
  }
  return out;
}
```

In [1]:
import math
import numpy as np

np.set_printoptions(precision=6, suppress=True)

def skew(v):
    x, y, z = v
    return np.array([
        [0.0, -z,  y],
        [z,   0.0, -x],
        [-y,  x,   0.0],
    ], dtype=float)

def exp_so3(phi):
    phi = np.asarray(phi, dtype=float).reshape(3)
    th = np.linalg.norm(phi)
    K = skew(phi)
    if th < 1e-12:
        return np.eye(3) + K + 0.5 * K @ K
    A = math.sin(th) / th
    B = (1.0 - math.cos(th)) / (th * th)
    return np.eye(3) + A * K + B * K @ K

def jr_so3(phi):
    phi = np.asarray(phi, dtype=float).reshape(3)
    th = np.linalg.norm(phi)
    K = skew(phi)
    if th < 1e-8:
        return np.eye(3) - 0.5 * K + (1.0/6.0) * K @ K
    A = (1.0 - math.cos(th)) / (th * th)
    B = (th - math.sin(th)) / (th ** 3)
    return np.eye(3) - A * K + B * K @ K

def R_impl(x):
    return exp_so3(x)

def boxplus_impl(x, y):
    return R_impl(y) @ x

def boxplus_jx_impl(x, y):
    return R_impl(y)

def boxplus_jy_impl(x, y):
    return -R_impl(y) @ skew(x) @ jr_so3(y)

def B_impl(x):
    x = np.asarray(x, dtype=float).reshape(3)
    e_i = np.array([0.0, 0.0, -1.0])
    E_jk = -np.eye(3)[:, :2]

    n = np.linalg.norm(x)
    if n < 1e-12:
        out = E_jk.copy()
    else:
        cross = np.cross(e_i, x)
        cross_norm = np.linalg.norm(cross)
        dot = float(np.dot(e_i, x))

        if cross_norm < 1e-12:
            if dot >= 0.0:
                out = E_jk.copy()
            else:
                out = R_impl(np.array([math.pi, 0.0, 0.0])) @ E_jk
        else:
            theta = math.atan2(cross_norm, dot)
            axis = cross / cross_norm
            out = R_impl(axis * theta) @ E_jk

    return out / n

def oplus_impl(x, u):
    x = np.asarray(x, dtype=float).reshape(3)
    u = np.asarray(u, dtype=float).reshape(2)
    if np.linalg.norm(u) < 1e-12:
        return x.copy()
    Bu = B_impl(x) @ u
    return R_impl(Bu) @ x

def oplus_ju_impl(x, u):
    x = np.asarray(x, dtype=float).reshape(3)
    u = np.asarray(u, dtype=float).reshape(2)
    Bx = B_impl(x)
    if np.linalg.norm(u) < 1e-12:
        return -skew(x) @ Bx
    Bu = Bx @ u
    return -R_impl(Bu) @ skew(x) @ jr_so3(Bu) @ Bx

def ominus_impl(y, x):
    x = np.asarray(x, dtype=float).reshape(3)
    y = np.asarray(y, dtype=float).reshape(3)
    cross = np.cross(x, y)
    cross_norm = np.linalg.norm(cross)
    dot = float(np.dot(x, y))

    if cross_norm >= 1e-12:
        axis = cross / cross_norm
        theta = math.atan2(cross_norm, dot)
        return B_impl(x).T @ (axis * theta)
    else:
        return np.zeros(2) if dot >= 0.0 else math.pi * np.array([1.0, 0.0])

def ominus_jy_impl(y, x):
    x = np.asarray(x, dtype=float).reshape(3)
    y = np.asarray(y, dtype=float).reshape(3)
    cross = np.cross(x, y)
    cross_norm = np.linalg.norm(cross)
    dot = float(np.dot(x, y))

    if cross_norm >= 1e-12:
        axis = cross / cross_norm
        cross_norm_sq = cross_norm**2
        y_norm_sq = float(np.dot(y, y))
        cross_outer = np.outer(cross, cross)
        J = B_impl(x).T @ (
            (1.0 / cross_norm)
            * (np.eye(3) - cross_outer / cross_norm_sq)
            @ skew(x)
            * math.atan2(cross_norm, dot)
            + np.outer(axis, (dot * y - y_norm_sq * x) / (cross_norm * y_norm_sq))
        )
        return J
    else:
        if dot < 0.0:
            raise ValueError("Derivative undefined at antipodal point.")
        return B_impl(x).T @ skew(x)

def finite_diff(fun, x, eps=1e-7):
    x = np.asarray(x, dtype=float)
    y0 = np.asarray(fun(x))
    J = np.zeros((y0.size, x.size))
    for i in range(x.size):
        d = np.zeros_like(x)
        d[i] = eps
        yp = np.asarray(fun(x + d)).reshape(-1)
        ym = np.asarray(fun(x - d)).reshape(-1)
        J[:, i] = (yp - ym) / (2.0 * eps)
    return J

## 4. Hàm `R(x)`

Code:

```cpp
inline Matrix3d R(const Vector3d& x) {
  return manif::SO3d::Tangent(x).exp().rotation();
}
```

Đây là helper đơn giản nhất:

\[
R(x)=\mathrm{Exp}_{SO(3)}(x).
\]

Nghĩa là:

- input: một vector trục-góc \(x\in\mathbb R^3\)
- output: ma trận quay \(R(x)\in SO(3)\)

Hàm này là viên gạch nền cho toàn bộ file, vì cả `boxplus` lẫn `oplus` đều thực chất là "quay một vector bởi một increment trên \(SO(3)\)".

## 5. `boxplus(x, y)`: helper quay một vector 3D bằng increment quay 3D

Code:

```cpp
Vector3d boxplus(const Vector3d& x, const Vector3d& y, ...)
{
  Matrix3d Ry = R(y);
  Vector3d out = Ry * x;

  if (J_x) *J_x = Ry;
  if (J_y) *J_y = -Ry * skew(x) * rjac(y);
  return out;
}
```

Về toán học, hàm này chỉ làm:

\[
\mathrm{boxplus}(x,y)=\mathrm{Exp}(y)\,x.
\]

Đây là lý do mình gọi nó là **helper ambient 3D**:

- nó nhận increment **3 chiều**
- nó không phải phép cập nhật 2D trên tangent plane của \(S^2\)

### Jacobian của `boxplus`

Theo đúng code:

\[
\frac{\partial\,\mathrm{Exp}(y)x}{\partial x} = \mathrm{Exp}(y),
\]

và

\[
\frac{\partial\,\mathrm{Exp}(y)x}{\partial y}
=
-\mathrm{Exp}(y)\,[x]_\times\,J_r(y),
\]

với \(J_r(y)\) là right Jacobian của \(SO(3)\).

Công thức thứ hai xuất hiện vì khi đổi một chút vector trục-góc \(y\), bạn không thể coi \(SO(3)\) là Euclid; phải đi qua right Jacobian.

In [2]:
# Kiểm tra số học cho Jacobian của boxplus
rng = np.random.default_rng(0)
x = rng.normal(size=3)
y = np.array([0.2, -0.3, 0.1])

Jx_analytic = boxplus_jx_impl(x, y)
Jy_analytic = boxplus_jy_impl(x, y)

Jx_fd = finite_diff(lambda xx: boxplus_impl(xx, y), x)
Jy_fd = finite_diff(lambda yy: boxplus_impl(x, yy), y)

print("Sai số max của J_x trong boxplus:", np.max(np.abs(Jx_analytic - Jx_fd)))
print("Sai số max của J_y trong boxplus:", np.max(np.abs(Jy_analytic - Jy_fd)))

Sai số max của J_x trong boxplus: 2.433907242416211e-10
Sai số max của J_y trong boxplus: 3.6013460269490594e-10


## 6. `B(x)`: dựng cơ sở tiếp tuyến tại \(x\)

Phần paper LIMOncello Appendix A viết ý tưởng như sau:

\[
R(x)=\mathrm{Exp}\!\left(
\frac{e_3\times x}{\|e_3\times x\|}
\operatorname{atan2}(\|e_3\times x\|, e_3^\top x)
\right),
\qquad
B(x)=R(x)\,[e_1\ e_2].
\]

Ý nghĩa hình học:

1. chọn một trục chuẩn
2. quay hệ chuẩn sao cho trục thứ ba trùng với \(x\)
3. lấy hai trục còn lại làm **basis của tangent plane**

### Nhưng code hiện thực khác paper ở 2 điểm

Trong code:

```cpp
Vector3d   e_i  = Vector3d::UnitZ() * -1.;
Matrix3x2d E_jk = Matrix3d::Identity().leftCols<2>() * -1.;
...
return out / x.norm();
```

Nên implementation thực tế là:

\[
B_{\text{code}}(x)
=
\frac{1}{\|x\|}\,\widetilde R(x)\,(-[e_1\ e_2]),
\]

với \(\widetilde R(x)\) được dựng từ trục tham chiếu \(-e_3\), không phải \(e_3\).

### Ý nghĩa của hai thay đổi này

**(a) Đổi dấu**  
Comment trong code ghi rõ: vì convention gravity ở project này khác với IKFoM gốc, nên basis bị "flip" để đồng nhất với cách gravity được lưu trong `State.hpp`.

**(b) Chia cho \(\|x\|\)**  
Điều này làm các cột của \(B(x)\) có norm bằng \(1/\|x\|\), không phải 1.  
Vì vậy:

\[
B(x)^\top x = 0,
\qquad
B(x)^\top B(x) = \frac{1}{\|x\|^2} I_2.
\]

Tức là \(B(x)\) vẫn là basis của tangent plane, nhưng là **basis đã scale**.

In [3]:
# Kiểm tra các tính chất hình học cơ bản của B(x)
xg = np.array([1.0, 2.0, 9.5])
Bx = B_impl(xg)

print("B(x)^T x =")
print(Bx.T @ xg)

print("\nB(x)^T B(x) =")
print(Bx.T @ Bx)

print("\n1 / ||x||^2 =")
print(1.0 / np.dot(xg, xg))

B(x)^T x =
[0. 0.]

B(x)^T B(x) =
[[0.010499 0.      ]
 [0.       0.010499]]

1 / ||x||^2 =
0.010498687664041995


## 7. `oplus(x, u)`: cập nhật một điểm trên \(S^2\) bằng increment 2D

Code:

```cpp
inline Vector3d oplus(const Vector3d& x, const Vector2d& u, ...)
{
  Matrix3d RBu = R(B(x)*u);
  Vector3d out = tinyu ? x : (RBu*x).eval();
  ...
}
```

Đây mới là phép cập nhật trên \(S^2\) tương ứng với Appendix A:

\[
x \oplus u = \mathrm{Exp}(B(x)u)\,x.
\]

Hiểu theo trực giác:

1. \(u\in\mathbb R^2\) là increment trong tangent plane tại \(x\)
2. \(B(x)u\in\mathbb R^3\) biến increment 2D đó thành một vector trục-góc 3D
3. \(\mathrm{Exp}(B(x)u)\) quay \(x\) trở lại mặt cầu

### Jacobian theo \(u\)

Code cài 2 trường hợp:

#### Khi \(u \approx 0\)

\[
\frac{\partial (x\oplus u)}{\partial u}\Big|_{u=0}
=
-[x]_\times B(x).
\]

Đó chính là dòng:

```cpp
*J_u = -skew(x) * B(x);
```

#### Khi \(u\) hữu hạn

\[
\frac{\partial (x\oplus u)}{\partial u}
=
-\mathrm{Exp}(B(x)u)\,[x]_\times\,J_r(B(x)u)\,B(x).
\]

Đó là dòng:

```cpp
*J_u = -RBu * skew(x) * rjac(B(x)*u) * B(x);
```

### Jacobian theo \(x\)

Code chỉ cài `J_x` tại \(u\approx 0\), và khi đó:

\[
\frac{\partial(x\oplus u)}{\partial x}\Big|_{u=0} = I_3.
\]

Điều này hợp lý vì trong filter hiện tại họ chỉ cần đúng trường hợp local quanh \(u=0\).

In [4]:
# Kiểm tra số học Jacobian theo u của oplus
x = np.array([1.0, 2.0, 9.5])
u = np.array([0.01, -0.02])

Ju_analytic = oplus_ju_impl(x, u)
Ju_fd = finite_diff(lambda uu: oplus_impl(x, uu), u)

print("Sai số max của J_u trong oplus:", np.max(np.abs(Ju_analytic - Ju_fd)))

Ju0_analytic = oplus_ju_impl(x, np.zeros(2))
Ju0_fd = finite_diff(lambda uu: oplus_impl(x, uu), np.zeros(2))

print("Sai số max của J_u tại u=0:", np.max(np.abs(Ju0_analytic - Ju0_fd)))

Sai số max của J_u trong oplus: 2.5401727110629935e-09
Sai số max của J_u tại u=0: 1.503675017833217e-09


## 8. `ominus(y, x)`: lấy sai khác 2D giữa hai điểm trên mặt cầu

Code:

```cpp
inline Vector2d ominus(const Vector3d& y, const Vector3d& x, ...)
{
  Vector3d cross      = x.cross(y);
  double   cross_norm = cross.norm();
  double   dot        = x.dot(y);

  if (cross_norm >= 1e-12) {
    Vector3d axis  = cross / cross_norm;
    double   theta = std::atan2(cross_norm, dot);
    out = B(x).transpose() * (axis*theta);
  } else {
    out = dot >= 0. ? Vector2d::Zero() : (M_PI*Vector2d::UnitX()).eval();
  }
}
```

Đây tương ứng với Appendix A:

\[
y \ominus x
=
B(x)^\top\!\left(
\theta \frac{x\times y}{\|x\times y\|}
\right),
\qquad
\theta = \operatorname{atan2}(\|x\times y\|, x^\top y).
\]

Ý nghĩa:

- \(x\times y\): trục quay ngắn nhất đưa \(x\) sang \(y\)
- \(\theta\): góc quay
- \(B(x)^\top\): kéo vector quay 3D đó về tọa độ 2D trên tangent plane tại \(x\)

### Jacobian theo \(y\)

Ở nhánh tổng quát, code cài trực tiếp công thức đạo hàm đầy đủ.  
Ở nhánh gần đồng hướng (\(\theta\to 0\)), paper cho xấp xỉ bậc một:

\[
y\ominus x \simeq B(x)^\top(x\times y),
\]

nên

\[
\frac{\partial(y\ominus x)}{\partial y}
\simeq
B(x)^\top [x]_\times.
\]

Đó chính là nhánh:

```cpp
*J_y = B(x).transpose() * skew(x);
```

### Trường hợp đối cực

Khi \(y=-x\), trục quay không duy nhất. Paper chọn quy ước:

\[
y\ominus x = \begin{bmatrix}\pi \\ 0\end{bmatrix}.
\]

Code cũng làm đúng vậy, và nếu còn đòi Jacobian ở ca này thì nó ném exception vì đạo hàm không xác định tốt.

In [5]:
# Kiểm tra số học Jacobian của ominus ở trường hợp tổng quát
x = np.array([1.0, 2.0, 9.5])
u = np.array([0.03, -0.05])
y = oplus_impl(x, u)

Jy_analytic = ominus_jy_impl(y, x)
Jy_fd = finite_diff(lambda yy: ominus_impl(yy, x), y)

print("Sai số max của J_y trong ominus:", np.max(np.abs(Jy_analytic - Jy_fd)))

# Kiểm tra nhánh gần đồng hướng: y = x
Jy_same = ominus_jy_impl(x, x)
Jy_same_formula = B_impl(x).T @ skew(x)

print("\nSai số nhánh y=x:", np.max(np.abs(Jy_same - Jy_same_formula)))

Sai số max của J_y trong ominus: 1.3025967891128198e-11

Sai số nhánh y=x: 0.0


## 9. Một tinh tế rất quan trọng: implementation này **không** lấy `oplus` và `ominus` làm inverse exact theo kiểu "cắm vào là ra lại nguyên xi"

Nếu đọc Appendix A theo trực giác đơn giản, ta dễ nghĩ:

\[
(x \oplus u)\ominus x \stackrel{?}{=} u.
\]

Nhưng trong implementation hiện tại, \(B(x)\) có thêm hệ số \(1/\|x\|\).  
Vì vậy, với đúng code này, đẳng thức trên **không phải** thứ được dùng như một inverse exact toàn cục.

Thứ mà filter thật sự cần là cặp Jacobian local:

\[
J_x = \left.\frac{\partial (y \ominus x)}{\partial y}\right|_{y=x}
= B(x)^\top [x]_\times,
\]

\[
J_u = \left.\frac{\partial (x \oplus u)}{\partial u}\right|_{u=0}
= -[x]_\times B(x).
\]

Điểm đẹp là hai ma trận này khớp nhau hoàn hảo:

\[
J_x J_u
=
B(x)^\top [x]_\times (-[x]_\times) B(x)
=
B(x)^\top\big(\|x\|^2 I - xx^\top\big)B(x)
=
I_2.
\]

Lý do là:

- \(B(x)^\top x = 0\)
- \(B(x)^\top B(x)=\frac{1}{\|x\|^2}I_2\)

Và chính **cặp Jacobian local này** mới là thứ `State.hpp` dùng để chuyển giữa:

- biểu diễn gravity 3D trong state
- biểu diễn gravity 2D trong covariance

In [6]:
# Kiểm tra thực nghiệm ý rất quan trọng ở trên
x = np.array([1.0, 2.0, 9.5])

Jx = B_impl(x).T @ skew(x)
Ju = -skew(x) @ B_impl(x)

print("Jx @ Ju =")
print(Jx @ Ju)

print("\nSai số so với I2:", np.max(np.abs(Jx @ Ju - np.eye(2))))

u = np.array([0.05, -0.02])
print("\nominus(oplus(x, u), x) =")
print(ominus_impl(oplus_impl(x, u), x))

print("\nu gốc =")
print(u)

Jx @ Ju =
[[1. 0.]
 [0. 1.]]

Sai số so với I2: 2.220446049250313e-16

ominus(oplus(x, u), x) =
[ 0.000525 -0.00021 ]

u gốc =
[ 0.05 -0.02]


### Ý nghĩa của kết quả ở cell trên

- `Jx @ Ju ≈ I₂`: đây mới là tính chất Kalman filter cần để local linearization của gravity nhất quán.
- `ominus(oplus(x,u),x)` **không bằng đúng** `u` với implementation có scaling hiện tại; đây không phải bug số học của notebook mà là hệ quả của cách `B(x)` được scale trong code.

Nói thẳng và thực tế:

> `S2.hpp` của project này được viết để phục vụ **cơ chế linearization của IESEKF**, không phải để làm một cặp retraction/lift "đẹp tuyệt đối" theo nghĩa hình thức cho mọi cách đọc trực giác.

## 10. `State.hpp` dùng các hàm này như thế nào?

Đây là các chỗ quan trọng trong `predict()`:

```cpp
Mat<2, 3> Jx;
S2::ominus(g(), g(), Jx);

Mat<3, 2> Ju;
S2::oplus(g(), {0., 0.}, {}, Ju);

Mat<DoFS2, DoF> left = Mat<DoFS2, DoF>::Identity();
left.bottomRightCorner<2, 3>() = Jx;

Mat<DoF, DoFS2> right = Mat<DoF, DoFS2>::Identity();
right.bottomRightCorner<3, 2>() = Ju;
```

Đọc theo nghĩa hình học:

- `Jx`: map từ **perturbation 3D của gravity đang lưu trong state** sang **2D tangent error**
- `Ju`: map ngược lại, từ **2D tangent correction** về **3D ambient update**

Nói đơn giản:

- `left` kéo gravity từ không gian lưu trữ 3D về không gian error-state 2D
- `right` đẩy correction 2D trở lại biểu diễn 3D

Còn trong `update()`:

```cpp
dx.tail(2) = S2::ominus(g(), g_pred);
...
g(S2::oplus(g(), dx.tail(2)));
```

Điều đó nghĩa là:

1. so gravity hiện tại với gravity prior bằng sai khác 2D
2. giải IESEKF trong không gian 2D đó
3. nạp correction 2D trở lại vector gravity 3D bằng `oplus`

Đây chính là chỗ `S2.hpp` nối trực tiếp vào thuật toán lọc.

## 11. `boxplus` được dùng trong `predict()` để làm gì?

Trong `State.hpp` còn có đoạn:

```cpp
Mat<3> AdjS2, JrS2;
S2::boxplus(g(), {0., 0., 0.}, AdjS2, JrS2);

Adj.bottomRightCorner<3, 3>() = AdjS2;
Jr.bottomRightCorner<3, 3>()  = JrS2;
```

Ở increment 0, từ công thức của `boxplus` ta có:

\[
\left.\frac{\partial\,\mathrm{boxplus}(x,y)}{\partial x}\right|_{y=0}
= I_3,
\qquad
\left.\frac{\partial\,\mathrm{boxplus}(x,y)}{\partial y}\right|_{y=0}
= -[x]_\times.
\]

Nên ở đây `boxplus` được dùng như một helper để lấy các block Jacobian 3x3 quanh gravity khi xây dựng ma trận chuyển trạng thái của prediction step.

Đó là lý do file này có cả:

- `boxplus`: helper 3D
- `oplus`, `ominus`: cặp phép toán 2D của gravity trên \(S^2\)

## 12. Những chỗ dễ hiểu nhầm nhất

### (1) `boxplus` không phải phép `⊕` của Appendix A
`boxplus(x, y3)` chỉ là helper quay vector bằng increment 3D.

### (2) `oplus` mới là phép cập nhật gravity bằng increment 2D
Đây là phép dùng thật ở cuối bước update:

```cpp
g(S2::oplus(g(), dx.tail(2)));
```

### (3) `ominus` trả ra vector 2D, không phải vector 3D
Vì tangent space của \(S^2\) có dimension 2.

### (4) `B(x)` trong code không trùng 100% với bản viết gọn trong paper
Implementation có:
- đổi dấu basis tham chiếu
- chia cho \(\|x\|\)

### (5) Điều IESEKF cần không phải là một cặp inverse global thật "đẹp"
Điều nó thật sự dùng là cặp Jacobian local `Jx`, `Ju`, và chúng khớp nhau tốt: `Jx @ Ju = I₂`.

## 13. Tóm tắt một dòng cho từng hàm

### `R(x)`
Biến vector trục-góc 3D thành ma trận quay.

### `boxplus(x, y3)`
Quay vector 3D `x` bằng increment quay 3D `y3`. Đây là helper ambient 3D.

### `B(x)`
Dựng basis 2D của tangent plane tại `x`, nhưng theo implementation đã:
- flip sign
- scale bởi \(1/\|x\|\)

### `oplus(x, u2)`
Nhận increment 2D trên tangent plane tại `x`, đổi nó thành quay 3D, rồi quay `x` để trở lại mặt cầu.

### `ominus(y, x)`
Lấy "sai khác ngắn nhất" từ `x` sang `y` rồi chiếu về tọa độ 2D trong tangent plane tại `x`.

---

Nếu phải nhớ đúng một câu:

> `S2.hpp` là bộ đổi tọa độ giữa **gravity 3D đang lưu trong state** và **gravity 2D mà IESEKF thật sự tối ưu trong covariance**.

## 14. Ghi chú cuối: nối paper với code theo đúng chỗ

Nếu bạn đọc paper và code song song, hãy nối như sau:

- **Appendix A Eq. (21a)**  → `S2::oplus`
- **Appendix A Eq. (21b)**  → `S2::ominus`
- **Appendix A Eq. (22)**   → ý tưởng dựng `B(x)`
- **Appendix A Eq. (23)**   → nhánh gần đồng hướng trong Jacobian của `ominus`
- **Appendix A Eq. (24)**   → nhánh đối cực trong `ominus`

Còn **`State.hpp`** cho bạn biết vì sao file này tồn tại trong toàn bộ backend:

- gravity được lưu là 3D
- error-state của gravity chỉ là 2D
- `S2.hpp` là nơi mang hình học \(S^2\) vào prediction và update của IESEKF